In [17]:
from lib import read_parquet_user, read_parquet_item, read_parquet_purchase, split_and_save_parquet, build_feature_label

# Cell 1 – Import & khai báo đường dẫn

In [18]:
import polars as pl
import numpy as np
import pandas as pd
import json

from tqdm import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Đường dẫn file train / eval
TRAIN_PATH = "./sale_pers.train_data_0.parquet"
EVAL_PATH  = "./sale_pers.eva_data_0.parquet"

# Top-k recommendation
K = 10

# Cell 2 – Load train_data & xem sơ

In [19]:
train_df = pl.read_parquet(TRAIN_PATH)
print("Train shape:", train_df.shape)
print(train_df)
print(train_df.schema)

Train shape: (7977870, 9)
shape: (7_977_870, 9)
┌────────────┬────────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬─────┐
│ customer_i ┆ item_id    ┆ brand_coun ┆ age_counts ┆ … ┆ segment_co ┆ target_us ┆ time_sinc ┆ Y   │
│ d          ┆ ---        ┆ ts         ┆ ---        ┆   ┆ unts       ┆ er_group_ ┆ e_last_pu ┆ --- │
│ ---        ┆ str        ┆ ---        ┆ u32        ┆   ┆ ---        ┆ counts    ┆ rchase_in ┆ i8  │
│ i32        ┆            ┆ u32        ┆            ┆   ┆ u32        ┆ ---       ┆ _B_…      ┆     │
│            ┆            ┆            ┆            ┆   ┆            ┆ u32       ┆ ---       ┆     │
│            ┆            ┆            ┆            ┆   ┆            ┆           ┆ i64       ┆     │
╞════════════╪════════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═════╡
│ 1923522    ┆ 5444000000 ┆ 3          ┆ 56         ┆ … ┆ 0          ┆ 81        ┆ 196       ┆ 0   │
│            ┆ 011        ┆            ┆   

# Cell 3 – Load eval_data & xem schema

In [20]:
eval_df = pl.read_parquet(EVAL_PATH)
print("Eval shape:", eval_df.shape)
print(eval_df.head())
print(eval_df.schema)

Eval shape: (24359444, 9)
shape: (5, 9)
┌────────────┬────────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬─────┐
│ customer_i ┆ item_id    ┆ brand_coun ┆ age_counts ┆ … ┆ segment_co ┆ target_us ┆ time_sinc ┆ Y   │
│ d          ┆ ---        ┆ ts         ┆ ---        ┆   ┆ unts       ┆ er_group_ ┆ e_last_pu ┆ --- │
│ ---        ┆ str        ┆ ---        ┆ u32        ┆   ┆ ---        ┆ counts    ┆ rchase_in ┆ i8  │
│ i32        ┆            ┆ u32        ┆            ┆   ┆ u32        ┆ ---       ┆ _B_…      ┆     │
│            ┆            ┆            ┆            ┆   ┆            ┆ u32       ┆ ---       ┆     │
│            ┆            ┆            ┆            ┆   ┆            ┆           ┆ i64       ┆     │
╞════════════╪════════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═════╡
│ 3511493    ┆ 0954000000 ┆ 2          ┆ 2          ┆ … ┆ 2          ┆ 2         ┆ 200       ┆ 0   │
│            ┆ 044        ┆            ┆           

# Cell 4 – Chuẩn bị X_train, y_train cho Logistic Regression

In [21]:
target_col = "Y"
drop_cols = ["customer_id", "item_id", target_col]

feature_cols = [c for c in train_df.columns if c not in drop_cols]

print("Số feature:", len(feature_cols))
print("Feature cols:", feature_cols)

if len(feature_cols) == 0:
    raise ValueError("Không có feature nào trong train_df ngoài customer_id, item_id, Y.")

X_train = train_df.select(feature_cols).to_pandas()
y_train = train_df[target_col].to_pandas()

X_train.shape, y_train.shape

Số feature: 6
Feature cols: ['brand_counts', 'age_counts', 'category_counts', 'segment_counts', 'target_user_group_counts', 'time_since_last_purchase_in_B_category']


((7977870, 6), (7977870,))

# Cell 5 – Khai báo mô hình Logistic (giống “scoring model” trong Recommender 101)

In [22]:
logreg_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "logreg",
            LogisticRegression(
                max_iter=500,
                class_weight="balanced",   # xử lý mất cân bằng nhãn
                solver="lbfgs",
                verbose=1,                 # in log train theo iteration
            ),
        ),
    ]
)

logreg_pipeline

,steps,"[('scaler', ...), ('logreg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


# Cell 6 – Train Logistic Regression trên train_data

In [23]:
print("=== BẮT ĐẦU TRAIN LOGISTIC REGRESSION ===")
logreg_pipeline.fit(X_train, y_train)
print("=== TRAIN XONG ===")


=== BẮT ĐẦU TRAIN LOGISTIC REGRESSION ===


=== TRAIN XONG ===


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    6.2s finished


# Cell 7 mới – Load eval_data (đầy đủ feature) & chuẩn bị X_eval, y_eval

In [24]:
# Đọc lại eval_data (full feature)
eval_df = pl.read_parquet(EVAL_PATH)
print("Eval shape:", eval_df.shape)
print(eval_df.head())
print(eval_df.schema)

target_col = "Y"

# Dùng cùng tập feature với train, nhưng kiểm tra xem eval có thiếu cột nào không
eval_feature_cols = [c for c in feature_cols if c in eval_df.columns]
missing_in_eval = set(feature_cols) - set(eval_feature_cols)

if missing_in_eval:
    print("[CẢNH BÁO] Eval thiếu các feature sau, sẽ bỏ qua khi predict:", missing_in_eval)

print("Số feature dùng cho eval:", len(eval_feature_cols))
print("Eval feature cols:", eval_feature_cols)

X_eval = eval_df.select(eval_feature_cols).to_pandas()
y_eval = eval_df[target_col].to_pandas()

cust_eval = eval_df["customer_id"].to_pandas().values
item_eval = eval_df["item_id"].to_pandas().values

X_eval.shape, y_eval.shape, len(cust_eval), len(item_eval)

Eval shape: (24359444, 9)
shape: (5, 9)
┌────────────┬────────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬─────┐
│ customer_i ┆ item_id    ┆ brand_coun ┆ age_counts ┆ … ┆ segment_co ┆ target_us ┆ time_sinc ┆ Y   │
│ d          ┆ ---        ┆ ts         ┆ ---        ┆   ┆ unts       ┆ er_group_ ┆ e_last_pu ┆ --- │
│ ---        ┆ str        ┆ ---        ┆ u32        ┆   ┆ ---        ┆ counts    ┆ rchase_in ┆ i8  │
│ i32        ┆            ┆ u32        ┆            ┆   ┆ u32        ┆ ---       ┆ _B_…      ┆     │
│            ┆            ┆            ┆            ┆   ┆            ┆ u32       ┆ ---       ┆     │
│            ┆            ┆            ┆            ┆   ┆            ┆           ┆ i64       ┆     │
╞════════════╪════════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═════╡
│ 3511493    ┆ 0954000000 ┆ 2          ┆ 2          ┆ … ┆ 2          ┆ 2         ┆ 200       ┆ 0   │
│            ┆ 044        ┆            ┆           

((24359444, 6), (24359444,), 24359444, 24359444)

# Cell 8 mới – Classification metrics trên tập eval (sanity check)

In [25]:
y_pred_eval = logreg_pipeline.predict(X_eval)
y_prob_eval = logreg_pipeline.predict_proba(X_eval)[:, 1]

print("=== Classification metrics trên EVAL (sanity check) ===")
auc_eval = roc_auc_score(y_eval, y_prob_eval)
print(f"ROC-AUC (eval): {auc_eval:.4f}\n")

print("Confusion matrix (eval):")
print(confusion_matrix(y_eval, y_pred_eval), "\n")

print("Classification report (eval):")
print(classification_report(y_eval, y_pred_eval))

=== Classification metrics trên EVAL (sanity check) ===
ROC-AUC (eval): 0.9856

Confusion matrix (eval):
[[20776476   973809]
 [   10367  2598792]] 

Classification report (eval):
              precision    recall  f1-score   support

           0       1.00      0.96      0.98  21750285
           1       0.73      1.00      0.84   2609159

    accuracy                           0.96  24359444
   macro avg       0.86      0.98      0.91  24359444
weighted avg       0.97      0.96      0.96  24359444



# Cell 9 mới – Hàm DCG & hàm build bảng top-K từ chính eval_data

In [26]:
def dcg_at_k(relevances: np.ndarray) -> float:
    """
    relevances: vector 0/1 theo thứ tự rank.
    """
    if len(relevances) == 0:
        return 0.0
    rel = np.asarray(relevances, dtype=float)
    discounts = np.log2(np.arange(2, len(rel) + 2))
    gains = (2.0**rel - 1.0)
    return float(np.sum(gains / discounts))


def build_user_topk_from_eval(
    customer_ids: np.ndarray,
    item_ids: np.ndarray,
    y_true: np.ndarray,
    scores: np.ndarray,
    k: int = 10,
) -> pd.DataFrame:
    """
    Tạo bảng mỗi dòng 1 user từ CHÍNH tập eval:

    Columns:
        - customer_id
        - top_k_items: dict {rank: item_id} với rank 1..K
        - precision_at_k
        - ndcg_at_k
    """
    df_eval_scores = pd.DataFrame(
        {
            "customer_id": customer_ids,
            "item_id": item_ids,
            "y_true": y_true,
            "score": scores,
        }
    )

    rows = []
    precisions = []
    ndcgs = []

    for uid, group in tqdm(
        df_eval_scores.groupby("customer_id"),
        desc=f"Tính top-{k}, Precision@{k}, NDCG@{k} (eval)",
        ncols=100,
    ):
        # sắp xếp item theo score giảm dần
        group_sorted = group.sort_values("score", ascending=False)

        if group_sorted.shape[0] == 0:
            continue

        # lấy top-k (nếu ít hơn K thì lấy hết)
        topk = group_sorted.head(k)
        actual_k = topk.shape[0]
        rel = topk["y_true"].values  # 0/1

        # Precision@K
        prec = rel.sum() / float(actual_k)
        precisions.append(prec)

        # NDCG@K
        dcg = dcg_at_k(rel)

        # Ideal DCG: sort theo y_true giảm dần, lấy K
        ideal_rel = np.sort(group["y_true"].values)[::-1][:k]
        idcg = dcg_at_k(ideal_rel)
        ndcg = dcg / idcg if idcg > 0 else 0.0
        ndcgs.append(ndcg)

        # dict rank -> item_id (KHÔNG chứa score, đúng format file nộp)
        topk_items_dict = {
            rank + 1: int(row.item_id)
            for rank, (_, row) in enumerate(topk.iterrows())
        }

        rows.append(
            {
                "customer_id": uid,
                "top_k_items": topk_items_dict,
                "precision_at_k": prec,
                "ndcg_at_k": ndcg,
            }
        )

    user_topk_eval_df = pd.DataFrame(rows)

    # In thêm global mean
    mean_prec = float(np.mean(precisions)) if precisions else 0.0
    mean_ndcg = float(np.mean(ndcgs)) if ndcgs else 0.0

    print(f"\n[GLOBAL] Mean Precision@{k} (eval): {mean_prec:.4f}")
    print(f"[GLOBAL] Mean NDCG@{k}     (eval): {mean_ndcg:.4f}")

    return user_topk_eval_df

# Cell 10 – Cell 10 mới – Chạy đánh giá top-K trên eval_data

In [27]:
user_topk_eval_df = build_user_topk_from_eval(
    customer_ids=cust_eval,
    item_ids=item_eval,
    y_true=y_eval.values,
    scores=y_prob_eval,
    k=K,
)

user_topk_eval_df.head()

Tính top-10, Precision@10, NDCG@10 (eval): 100%|████████| 2438242/2438242 [21:22<00:00, 1900.56it/s]



[GLOBAL] Mean Precision@10 (eval): 0.1402
[GLOBAL] Mean NDCG@10     (eval): 0.2399


,customer_id,top_k_items,precision_at_k,ndcg_at_k
0,14732,"{1: 5468000000001, 2: 1386000000008, 3: 291400...",0.0,0.0
1,15126,"{1: 7155000000001, 2: 1237000000008, 3: 582600...",0.0,0.0
2,17212,"{1: 3775000000003, 2: 2584000000003, 3: 520400...",0.0,0.0
3,17224,{1: 175000000011},0.0,0.0
4,17266,{1: 2414000000003},0.0,0.0


# Cell 16 – Xuất file CSV nội bộ (có thêm 2 cột metric)

In [30]:
debug_eval_df = user_topk_eval_df.copy()
debug_eval_df["top_k_items"] = debug_eval_df["top_k_items"].apply(json.dumps)

debug_eval_path = "./eval_recommend_debug_with_metrics.csv"
debug_eval_df.to_csv(debug_eval_path, index=False, encoding="utf-8")

print("Đã lưu file nội bộ (eval, có metric) tại:", debug_eval_path)
debug_eval_df.head()

Đã lưu file nội bộ (eval, có metric) tại: ./eval_recommend_debug_with_metrics.csv


,customer_id,top_k_items,precision_at_k,ndcg_at_k
0,14732,"{""1"": 5468000000001, ""2"": 1386000000008, ""3"": ...",0.0,0.0
1,15126,"{""1"": 7155000000001, ""2"": 1237000000008, ""3"": ...",0.0,0.0
2,17212,"{""1"": 3775000000003, ""2"": 2584000000003, ""3"": ...",0.0,0.0
3,17224,"{""1"": 175000000011}",0.0,0.0
4,17266,"{""1"": 2414000000003}",0.0,0.0


# Cell 17 – Xuất file CSV nộp cho giảng viên (CHỈ 2 cột)

In [31]:
submit_eval_df = user_topk_eval_df[["customer_id", "top_k_items"]].copy()
submit_eval_df["top_k_items"] = submit_eval_df["top_k_items"].apply(json.dumps)

submit_eval_path = "./submission_topk_from_eval.csv"
submit_eval_df.to_csv(submit_eval_path, index=False, encoding="utf-8")

print("ĐÃ TẠO FILE NỘP (dựa trên eval) TẠI:", submit_eval_path)
submit_eval_df.head()


ĐÃ TẠO FILE NỘP (dựa trên eval) TẠI: ./submission_topk_from_eval.csv


,customer_id,top_k_items
0,14732,"{""1"": 5468000000001, ""2"": 1386000000008, ""3"": ..."
1,15126,"{""1"": 7155000000001, ""2"": 1237000000008, ""3"": ..."
2,17212,"{""1"": 3775000000003, ""2"": 2584000000003, ""3"": ..."
3,17224,"{""1"": 175000000011}"
4,17266,"{""1"": 2414000000003}"


# Kiểm tra tính đúng đắn

In [33]:
uid = user_topk_eval_df["customer_id"].iloc[9]  # ví dụ lấy user đầu tiên

# Lấy top-k items của user trong bảng top-k
row = user_topk_eval_df[user_topk_eval_df["customer_id"] == uid].iloc[0]
topk_dict = row["top_k_items"]  # dict rank -> item_id

top_items = [topk_dict[r] for r in sorted(topk_dict.keys())]

print("User:", uid)
print("Top-10 items:", top_items)

# Lấy ground truth y_true cho user trên eval_data
user_eval = eval_df.filter(pl.col("customer_id") == uid).to_pandas()

# Map item_id -> y_true
truth_map = {
    int(r.item_id): int(r.Y)
    for _, r in user_eval.iterrows()
}

rel = [truth_map.get(itm, 0) for itm in top_items]
print("Relevance (rel):", rel)


User: 28879
Top-10 items: [2678000000002, 2700000000002, 6004000000005, 5006000000008, 4950000000002, 4950000000001, 7090000157, 68000000168, 68000000033, 68000000159]
Relevance (rel): [1, 1, 1, 1, 0, 1, 1, 1, 1, 1]


In [34]:
import numpy as np

def dcg_at_k_np(relevances):
    rel = np.asarray(relevances, dtype=float)
    if len(rel) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(rel) + 2))
    gains = (2.0**rel - 1.0)
    return float(np.sum(gains / discounts))

rel_arr = np.array(rel)

prec_manual = rel_arr.sum() / len(rel_arr)
dcg_manual = dcg_at_k_np(rel_arr)

# Ideal rel: sort y_true của toàn bộ candidate theo giảm dần
all_rel = np.sort(user_eval["Y"].values)[::-1][:len(rel_arr)]
idcg_manual = dcg_at_k_np(all_rel)
ndcg_manual = dcg_manual / idcg_manual if idcg_manual > 0 else 0.0

print("PRECISION@10 manual:", prec_manual)
print("NDCG@10 manual     :", ndcg_manual)

print("PRECISION@10 stored:", row["precision_at_k"])
print("NDCG@10 stored     :", row["ndcg_at_k"])

PRECISION@10 manual: 0.9
NDCG@10 manual     : 0.9148568823583791
PRECISION@10 stored: 0.9
NDCG@10 stored     : 0.9148568823583791


Dùng thư viện ngoài để đối chiếu

In [35]:
from sklearn.metrics import ndcg_score

# Cho 1 user: y_true (0/1) cho tất cả candidate items, y_score = score model
user_eval_df = eval_df.filter(pl.col("customer_id") == uid).to_pandas()

y_true_full = user_eval_df["Y"].values.reshape(1, -1)
y_score_full = logreg_pipeline.predict_proba(
    user_eval_df[eval_feature_cols]
)[:, 1].reshape(1, -1)

ndcg_sklearn = ndcg_score(y_true_full, y_score_full, k=10)
print("NDCG@10 (sklearn) :", ndcg_sklearn)
print("NDCG@10 (manual)  :", ndcg_manual)

NDCG@10 (sklearn) : 0.9182293066898518
NDCG@10 (manual)  : 0.9148568823583791


So sánh với baseline ngẫu nhiên

In [36]:
# Tạo random score cho eval
np.random.seed(0)
random_scores = np.random.rand(len(eval_df))

# Tính top-k từ random như với model
user_topk_random_df = build_user_topk_from_eval(
    customer_ids=eval_df["customer_id"].to_pandas().values,
    item_ids=eval_df["item_id"].to_pandas().values,
    y_true=eval_df["Y"].to_pandas().values,
    scores=random_scores,
    k=10,
)

Tính top-10, Precision@10, NDCG@10 (eval): 100%|████████| 2438242/2438242 [21:41<00:00, 1874.10it/s]



[GLOBAL] Mean Precision@10 (eval): 0.1025
[GLOBAL] Mean NDCG@10     (eval): 0.1361


Khi so sánh với baseline ngẫu nhiên cho thấy `mean` của 2 độ đo khi cho mô hình học `với score được tính toán cẩn thận cho eval` so với `random score cho tập eval` thì nó lớn hơn => mô hình học được đặc trưng => ý nghĩa hơn